In [146]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai._function_utils import _convert_pydantic_to_genai_function
load_dotenv()
google_api_key = os.getenv('GOOGLE_API_KEY')
chat = ChatGoogleGenerativeAI(temperature = 0, model = 'gemini-2.0-flash')


In [147]:
from pydantic import BaseModel, Field
# Function to convert a Pydantic model to the Gemini tool format
def pydantic_to_gemini_tool(model: BaseModel, function_name: str = None) -> dict:
    """
    Converts a Pydantic BaseModel into a dictionary representing a Gemini function tool.

    Args:
        model: The Pydantic BaseModel defining the function's parameters.
        function_name: Optional. The name of the function. If not provided,
                       the Pydantic model's class name will be used (converted to snake_case).

    Returns:
        A dictionary formatted as a Gemini function tool.
    """
    if function_name is None:
        # Convert CamelCase Pydantic model name to snake_case function name
        function_name = ''.join(['_' + i.lower() if i.isupper() else i for i in model.__name__]).lstrip('_')

    return {
        "function_declarations": [
            {
                "name": function_name,
                "description": model.__doc__.strip() if model.__doc__ else "",
                "parameters": model.model_json_schema(),
            }
        ]
    }

In [148]:
from langchain.schema.output_parser import StrOutputParser
from langchain.output_parsers import JsonOutputToolsParser
from langchain.prompts import ChatPromptTemplate
class Tagging(BaseModel):
    """We will be using tagging to determine the sentiments of the sentence and language."""
    sentiment: str = Field(description="Classify whether the text is positive, negative or neutral.")
    language: str = Field(description="the language of the text.(should be ISO 639-1 code)")
tools = [_convert_pydantic_to_genai_function(Tagging)]
model = chat.bind_tools([Tagging])

In [149]:
prompt = ChatPromptTemplate.from_messages([("system","Think carefully, and tag the text as instructed. Always use the tagging tool for sentiment and language analysis."), ("user", "{input}")])
chain = prompt | model | JsonOutputToolsParser()
res = chain.invoke({'input': "The glaciers in the Himalayas are getting melt down due to the Global Warming."})

In [150]:
print(res)

[{'args': {'language': 'en', 'sentiment': 'negative'}, 'type': 'Tagging'}]


In [151]:
print(chain.invoke({'input': "El higo de la puta"}))

[{'args': {'language': 'es', 'sentiment': 'negative'}, 'type': 'Tagging'}]


In [152]:
print(chain.invoke({'input': "Cristiano Ronaldo DOs Santos Aveiro es el mejor."}))

[{'args': {'language': 'es', 'sentiment': 'positive'}, 'type': 'Tagging'}]


In [153]:
print(chain.invoke({'input': "Bend it like Beckham"}))

[{'args': {'language': 'en', 'sentiment': 'positive'}, 'type': 'Tagging'}]


In [154]:
print(model.invoke("Tell me the sentiment and language of the sentence: Portugal wins the FIFA World Cup 2026."))

content='' additional_kwargs={'function_call': {'name': 'Tagging', 'arguments': '{"language": "en", "sentiment": "positive"}'}} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []} id='run--96d9df4b-2365-49f0-920f-e8b8e58686ab-0' tool_calls=[{'name': 'Tagging', 'args': {'language': 'en', 'sentiment': 'positive'}, 'id': 'f8a5b9d8-769b-4a95-846b-578b097c5035', 'type': 'tool_call'}] usage_metadata={'input_tokens': 74, 'output_tokens': 6, 'total_tokens': 80, 'input_token_details': {'cache_read': 0}}
